<a href="https://colab.research.google.com/github/zackdihel/DA-final-project/blob/main/notebooks/educ_cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [29]:
import pandas as pd

In [30]:
df = pd.read_csv('../data/educ.csv')
print(f"  Raw N: {len(df):,}")

  Raw N: 1,777,193


In [31]:
df = df[(df['AGE'] >= 25) & (df['AGE'] <= 64)]  # working-age only
df = df[df['EMPSTAT'].isin([1, 2])]    # employed or unemployed
df = df[df['INCTOT'] != 9999999]  #drop missing income
print(f"  After filtering age, labor force status, and missing income: {len(df):,}")

  After filtering age, labor force status, and missing income: 759,980


In [32]:
# EMPSTAT: 1 = employed, 2 = unemployed
df['unemployed'] = (df['EMPSTAT'] == 2).astype(int)

In [33]:
# EDUCD >= 101: bachelor's, master's, professional, or PhD
df['college_degree'] = (df['EDUCD'] >= 101).astype(int)

In [34]:
# STATEFIP: 25 = Massachusetts, 33 = New Hampshire  # 25 and 33 are original state designations from IPUMS
df['is_MA'] = (df['STATEFIP'] == 25).astype(int)

In [35]:
df = df.drop(columns=['SAMPLE', 'SERIAL', 'CBSERIAL', 'HHWT',
                       'CLUSTER', 'STRATA', 'GQ', 'PERNUM',
                       'EMPSTATD', 'HISPAND', 'RACED'])

In [36]:
print(f"\n  Final N:              {len(df):,}")
print(f"  MA N:                 {(df['STATEFIP']==25).sum():,}")
print(f"  NH N:                 {(df['STATEFIP']==33).sum():,}")
print(f"  Unemployment rate:    {df['unemployed'].mean()*100:.2f}%")
print(f"  College degree rate:  {df['college_degree'].mean()*100:.2f}%")


  Final N:              759,980
  MA N:                 623,944
  NH N:                 136,036
  Unemployment rate:    4.60%
  College degree rate:  49.67%


In [37]:
output_path = '../data/educ_cleaned.csv'
df.to_csv(output_path, index=False)
print(f"\n  Saved to: {output_path}")


  Saved to: ../data/educ_cleaned.csv


In [38]:
df = pd.read_csv('../data/educ_cleaned.csv')
print(f"IPUMS data loaded: {len(df):,} rows")
print(f"Years in IPUMS: {df['YEAR'].min()} - {df['YEAR'].max()}")

IPUMS data loaded: 759,980 rows
Years in IPUMS: 2000 - 2024


In [39]:
#tuition data, this will be our instrument later on
shef_data = {
    'YEAR': list(range(2000, 2025)) * 2,
    'STATEFIP': [25]*25 + [33]*25,
    'tuition_per_fte': [
        # Massachusetts (STATEFIP 25)
        3504.86, 3402.54, 3435.84, 3868.98, 4675.30,
        5041.93, 5241.19, 5479.32, 5756.80, 5878.49,
        5984.10, 6322.29, 5566.53, 5842.39, 5919.58,
        6030.90, 6335.97, 6712.69, 7324.31, 6720.80,
        7013.36, 6967.12, 6668.42, 9539.09, 8370.01,
        # New Hampshire (STATEFIP 33)
        6208.86, 7300.23, 7474.05, 5624.39, 4936.20,
        6251.93, 6516.34, 7677.22, 8097.87, 8247.85,
        8034.88, 8753.03, 9566.79, 10182.09, 10546.24,
        10555.41, 10295.78, 10471.04, 10589.78, 10992.48,
        11135.89, 11272.51, 11357.97, 11539.08, 11421.22,
    ]
}

tuition = pd.DataFrame(shef_data)
print(f"\nSHEF tuition table built: {len(tuition)} rows (25 years x 2 states)")
print(tuition.groupby('STATEFIP')['tuition_per_fte'].describe().round(2))


SHEF tuition table built: 50 rows (25 years x 2 states)
          count     mean      std      min      25%      50%       75%  \
STATEFIP                                                                 
25         25.0  5904.11  1460.91  3402.54  5241.19  5919.58   6712.69   
33         25.0  9001.97  2073.89  4936.20  7474.05  9566.79  10589.78   

               max  
STATEFIP            
25         9539.09  
33        11539.08  


In [40]:
df = df.merge(tuition, on=['YEAR', 'STATEFIP'], how='left')

In [41]:
nulls = df['tuition_per_fte'].isnull().sum()
print(f"\nMerge complete: {len(df):,} rows")
print(f"Nulls in tuition_per_fte: {nulls}")
print(f"\nSample — tuition values by state and year:")
print(df.groupby(['STATEFIP', 'YEAR'])['tuition_per_fte'].first().unstack().T.head(10).round(2))


Merge complete: 759,980 rows
Nulls in tuition_per_fte: 0

Sample — tuition values by state and year:
STATEFIP       25       33
YEAR                      
2000      3504.86  6208.86
2001      3402.54  7300.23
2002      3435.84  7474.05
2003      3868.98  5624.39
2004      4675.30  4936.20
2005      5041.93  6251.93
2006      5241.19  6516.34
2007      5479.32  7677.22
2008      5756.80  8097.87
2009      5878.49  8247.85


In [42]:
df.to_csv('../data/educ_cleaned_instr.csv', index=False)